In [5]:
pip install PyPDF2 python-docx pandas ollama langchain langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
ollama pull llama3.2

SyntaxError: invalid syntax (1955164888.py, line 1)

In [6]:
!pip install PyPDF2 python-docx pandas ollama langchain langchain-community

In [7]:
import pandas as pd
import ollama

df = pd.read_csv("insurance_claim_rejections_dataset.csv")
print(df.shape)
print(df.columns.tolist())

(1000, 7)
['Claim_ID', 'Claim_Type', 'Rejection_Reason', 'Rejection_Letter_Text', 'Policy_Clause', 'Plain_English_Explanation', 'Appeal_Suggestion']


In [8]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

print("PDF extractor ready!")


PDF extractor ready!


In [9]:
def match_from_dataset(claim_type, rejection_text):
    filtered = df[df['Claim_Type'].str.lower() == claim_type.lower()]
    for _, row in filtered.iterrows():
        if any(word in rejection_text.lower() for word in row['Rejection_Reason'].lower().split()):
            return row
    return filtered.iloc[0]  # fallback to first match

print("Dataset matcher ready!")

Dataset matcher ready!


In [10]:
def explain_and_appeal(rejection_text, claim_type):
    matched = match_from_dataset(claim_type, rejection_text)
    
    prompt = f"""
You are an insurance expert. Analyze this rejection letter and help the policyholder.

Rejection Letter:
{rejection_text}

Known Rejection Reason: {matched['Rejection_Reason']}
Policy Clause: {matched['Policy_Clause']}
Plain English Explanation: {matched['Plain_English_Explanation']}
Appeal Suggestion: {matched['Appeal_Suggestion']}

1. Explain the rejection in very simple plain language.
2. Write a formal appeal letter the policyholder can send.
"""
    
    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content']

print("AI function ready!")

AI function ready!


In [12]:
!pip install -U langchain langchain-community

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 4.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.4 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 7.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.2 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.2.28

   ---- ----------------------------------- 1/9 [langchain-core]
    Uninstalling langchain-core-1.2.28:
   ---- ----------------------------------- 1/9 [langchain-core]
      Successfully uninstalled langchain-core-1.2.28
   ---- ----------------------------------- 1/9 [langchain-core]
   ---- ----------------------------------- 1/9 [langchain-core]
   ---- ----------------------------------

In [13]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = Ollama(model="llama3")

prompt_template = PromptTemplate(
    input_variables=["rejection_text", "rejection_reason", "policy_clause", "plain_explanation", "appeal_suggestion"],
    template="""
You are an insurance expert helping a policyholder.

Rejection Letter:
{rejection_text}

Known Rejection Reason: {rejection_reason}
Policy Clause: {policy_clause}
Plain English Explanation: {plain_explanation}
Appeal Suggestion: {appeal_suggestion}

1. Explain the rejection in very simple plain language (3-4 lines).
2. Write a formal structured appeal letter.
"""
)

chain = LLMChain(llm=llm, prompt=prompt_template)

def explain_and_appeal(rejection_text, claim_type):
    matched = match_from_dataset(claim_type, rejection_text)
    result = chain.run(
        rejection_text=rejection_text,
        rejection_reason=matched['Rejection_Reason'],
        policy_clause=matched['Policy_Clause'],
        plain_explanation=matched['Plain_English_Explanation'],
        appeal_suggestion=matched['Appeal_Suggestion']
    )
    return result, matched

print("LangChain + Ollama ready!")

ModuleNotFoundError: No module named 'langchain.prompts'

In [14]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = Ollama(model="llama3")

prompt_template = PromptTemplate(
    input_variables=["rejection_text", "rejection_reason", "policy_clause", "plain_explanation", "appeal_suggestion"],
    template="""
You are an insurance expert helping a policyholder.

Rejection Letter:
{rejection_text}

Known Rejection Reason: {rejection_reason}
Policy Clause: {policy_clause}
Plain English Explanation: {plain_explanation}
Appeal Suggestion: {appeal_suggestion}

1. Explain the rejection in very simple plain language (3-4 lines).
2. Write a formal structured appeal letter.
"""
)

chain = LLMChain(llm=llm, prompt=prompt_template)

def explain_and_appeal(rejection_text, claim_type):
    matched = match_from_dataset(claim_type, rejection_text)
    result = chain.run(
        rejection_text=rejection_text,
        rejection_reason=matched['Rejection_Reason'],
        policy_clause=matched['Policy_Clause'],
        plain_explanation=matched['Plain_English_Explanation'],
        appeal_suggestion=matched['Appeal_Suggestion']
    )
    return result, matched

print("LangChain + Ollama ready!")

ModuleNotFoundError: No module named 'langchain.chains'

In [15]:
!pip install langchain-core langchain-ollama

In [16]:
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = Ollama(model="llama3")

prompt_template = PromptTemplate(
    input_variables=["rejection_text", "rejection_reason", "policy_clause", "plain_explanation", "appeal_suggestion"],
    template="""
You are an insurance expert helping a policyholder.

Rejection Letter:
{rejection_text}

Known Rejection Reason: {rejection_reason}
Policy Clause: {policy_clause}
Plain English Explanation: {plain_explanation}
Appeal Suggestion: {appeal_suggestion}

1. Explain the rejection in very simple plain language (3-4 lines).
2. Write a formal structured appeal letter.
"""
)

chain = prompt_template | llm | StrOutputParser()

def explain_and_appeal(rejection_text, claim_type):
    matched = match_from_dataset(claim_type, rejection_text)
    result = chain.invoke({
        "rejection_text": rejection_text,
        "rejection_reason": matched['Rejection_Reason'],
        "policy_clause": matched['Policy_Clause'],
        "plain_explanation": matched['Plain_English_Explanation'],
        "appeal_suggestion": matched['Appeal_Suggestion']
    })
    return result, matched

print("LangChain + Ollama ready!")


LangChain + Ollama ready!


C:\Users\HP\AppData\Local\Temp\ipykernel_19772\2024515649.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")


In [17]:
from docx import Document

def save_as_docx(content, claim_type, filename="appeal_letter.docx"):
    doc = Document()
    doc.add_heading("Insurance Claim Appeal Letter", 0)
    doc.add_paragraph(f"Claim Type: {claim_type}")
    doc.add_paragraph("---")
    for line in content.split("\n"):
        if line.strip():
            doc.add_paragraph(line)
    doc.save(filename)
    print(f"Saved: {filename}")

print("DOCX generator ready!")

DOCX generator ready!


In [18]:
# Test rejection letters for all 3 types

health_rejection = """
Dear Policyholder, your claim for hospitalization at City Hospital has been rejected. 
The treatment was received at a non-network hospital without prior emergency authorization. 
As per your policy terms, full reimbursement is only applicable at network hospitals.
"""

motor_rejection = """
Dear Policyholder, your motor insurance claim has been rejected.
The vehicle was being driven without a valid driving license at the time of the accident.
As per policy clause 5.2, claims are void if the driver does not hold a valid license.
"""

life_rejection = """
Dear Policyholder, the death benefit claim has been rejected.
The life assured had stopped paying premiums and the policy lapsed before acquiring paid-up value.
As per clause 3.4, no benefit is payable if premiums were not paid for at least two years.
"""

print("Test letters ready!")

Test letters ready!


In [19]:
# Test Health
print("=== HEALTH CLAIM ===")
result_health, matched_health = explain_and_appeal(health_rejection, "Health")
print(result_health)
save_as_docx(result_health, "Health", "health_appeal.docx")

=== HEALTH CLAIM ===
**Simple Plain Language Explanation:**

Your claim for hospitalization at City Hospital has been rejected because you went to a non-network hospital without prior emergency authorization. This means that your specific plan doesn't cover out-of-network care, and we can only reimburse you up to a certain limit or exclude the claim altogether.

**Formal Structured Appeal Letter:**

[Policyholder's Name]
[Policyholder's Address]
[City, State, ZIP]
[Email Address]
[Date]

Insurance Company's Claims Department
[Insurance Company's Address]
[City, State, ZIP]

Dear Claims Review Committee,

I am writing to appeal the rejection of my claim for hospitalization at City Hospital. I received a letter dated [Date] informing me that my treatment was received at a non-network hospital without prior emergency authorization, and therefore, my claim has been rejected.

As per our policy terms, Section 3.1 states that cashless facilities and full reimbursements are only applicable at

In [20]:
print("=== MOTOR CLAIM ===")
result_motor, matched_motor = explain_and_appeal(motor_rejection, "Motor")
print(result_motor)
save_as_docx(result_motor, "Motor", "motor_appeal.docx")

=== MOTOR CLAIM ===
Here are the responses:

**Simple Plain Language Explanation:**

Your motor insurance claim was rejected because you were driving your car without a valid driver's license at the time of the accident. Additionally, the car was being used for commercial purposes, which is not covered by your personal auto policy.

**Formal Structured Appeal Letter:**

[Date]

Insurance Company
[Address]
[City, State, ZIP]
[Email Address]
[Phone Number]

Dear Insurance Claims Appeals Committee,

Re: Rejection of Motor Insurance Claim - [Policyholder's Name]

I am writing to appeal the rejection of my motor insurance claim dated [Claim Date]. The rejection letter stated that the claim was rejected due to the vehicle being driven without a valid driving license at the time of the accident, and commercial use of a personal vehicle.

However, I would like to provide additional information to support my claim. Specifically, I had obtained a valid driver's license on [License Date] and had 

In [21]:
print("=== LIFE CLAIM ===")
result_life, matched_life = explain_and_appeal(life_rejection, "Life")
print(result_life)
save_as_docx(result_life, "Life", "life_appeal.docx")

=== LIFE CLAIM ===
**Simple Plain Language Explanation**

Your death benefit claim has been rejected because you stopped paying premiums and your policy lapsed before it could gain paid-up value. This means that, according to our policy terms, there is no payout value for the policy.

**Formal Structured Appeal Letter**

[Date]

Grievance Department
[Insurance Company Name]
[Address]
[City, State, ZIP]

Dear Grievance Committee,

Re: Rejection of Death Benefit Claim (Claim Number: [insert claim number])

I am writing to express my disappointment and concern regarding the rejection of my death benefit claim. As per your rejection letter dated [insert date], the claim was denied due to a lapse in premiums and failure to acquire paid-up value, as stated in Clause 3.4 of our policy.

However, I would like to bring to your attention that there may have been an error on behalf of the insurance company that prevented timely payment of premiums. Specifically, [insert specific details of any at

In [22]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

# Test with a real PDF rejection letter
# Place any PDF in your notebook folder and change the filename
pdf_text = extract_text_from_pdf("rejection_letter.pdf")
print(pdf_text)

FileNotFoundError: [Errno 2] No such file or directory: 'rejection_letter.pdf'

In [23]:
from fpdf import FPDF
import subprocess
subprocess.run(["pip", "install", "fpdf2"], capture_output=True)

from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(0, 10, """Dear Policyholder,

We regret to inform you that your health insurance claim has been rejected.

Your claim for hospitalization at City Hospital has been denied because the 
treatment was received at a non-network hospital without prior emergency 
authorization. As per your policy terms Section 3.1, full reimbursement is 
only applicable at network hospitals.

We hope you understand our policy terms.

Regards,
Claims Department""")

pdf.output("rejection_letter.pdf")
print("Sample rejection PDF created!")

ModuleNotFoundError: No module named 'fpdf'

In [24]:
!pip install fpdf2

In [25]:
from fpdf import FPDF
import subprocess
subprocess.run(["pip", "install", "fpdf2"], capture_output=True)

from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(0, 10, """Dear Policyholder,

We regret to inform you that your health insurance claim has been rejected.

Your claim for hospitalization at City Hospital has been denied because the 
treatment was received at a non-network hospital without prior emergency 
authorization. As per your policy terms Section 3.1, full reimbursement is 
only applicable at network hospitals.

We hope you understand our policy terms.

Regards,
Claims Department""")

pdf.output("rejection_letter.pdf")
print("Sample rejection PDF created!")

Sample rejection PDF created!


In [26]:
pdf_text = extract_text_from_pdf("rejection_letter.pdf")
print(pdf_text)

Dear Policyholder,
We regret to inform you that your health insurance claim has been rejected.
Your claim for hospitalization at City Hospital has been denied because the 
treatment was received at a non-network hospital without prior emergency 
authorization. As per your policy terms Section 3.1, full reimbursement is 
only applicable at network hospitals.
We hope you understand our policy terms.
Regards,
Claims Department


In [27]:
# Full pipeline using actual PDF
result, matched = explain_and_appeal(pdf_text, "Health")
print(result)
save_as_docx(result, "Health", "pdf_appeal_output.docx")

**Simple Explanation**

Your health insurance claim was rejected because you got treated at a hospital that's not on our list of approved providers. Your plan doesn't cover care from out-of-network hospitals, so we can't reimburse the full amount.

**Formal Appeal Letter**

[Policyholder's Name]
[Address]
[City, State, ZIP]
[Email Address]
[Phone Number]
[Date]

Claims Department
[Insurer's Name]
[Address]
[City, State, ZIP]

Dear Claims Committee,

I am writing to appeal the rejection of my health insurance claim for hospitalization at City Hospital. As outlined in the rejection letter, the reason given was that the treatment was received at a non-network hospital without prior emergency authorization.

Although I understand that my policy terms specify that full reimbursement is only applicable at network hospitals (Section 3.1), I would like to request an exception based on the circumstances of my case. On [date of admission], I presented with a life-threatening medical emergency th

In [28]:
from fpdf import FPDF

# Health rejection
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(0, 10, """Dear Policyholder,

We regret to inform you that your health insurance claim has been rejected.

Your claim for hospitalization at City Hospital has been denied because the 
treatment was received at a non-network hospital without prior emergency 
authorization. As per your policy terms Section 3.1, full reimbursement is 
only applicable at network hospitals.

Regards,
Claims Department""")
pdf.output("D:\\wrwrw\\health_rejection.pdf")

# Motor rejection
pdf2 = FPDF()
pdf2.add_page()
pdf2.set_font("Helvetica", size=12)
pdf2.multi_cell(0, 10, """Dear Policyholder,

We regret to inform you that your motor insurance claim has been rejected.

Your vehicle was being driven without a valid driving license at the time 
of the accident. As per policy clause 5.2, claims are void if the driver 
does not hold a valid license at the time of the incident.

Regards,
Claims Department""")
pdf2.output("D:\\wrwrw\\motor_rejection.pdf")

# Life rejection
pdf3 = FPDF()
pdf3.add_page()
pdf3.set_font("Helvetica", size=12)
pdf3.multi_cell(0, 10, """Dear Policyholder,

We regret to inform you that your life insurance death benefit claim has been rejected.

The life assured had stopped paying premiums and the policy lapsed before 
acquiring paid-up value. As per clause 3.4, no benefit is payable if premiums 
were not paid for at least two consecutive years.

Regards,
Claims Department""")
pdf3.output("D:\\wrwrw\\life_rejection.pdf")

print("All 3 rejection PDFs saved to D:\\wrwrw!")

All 3 rejection PDFs saved to D:\wrwrw!
